# 12.3 椭圆型偏微分方程

椭圆型方程通常由边界条件决定整个区域内的解。有限差分离散后得到稀疏线性系统，可以用 Jacobi、Gauss-Seidel、SOR 或直接法求解。

In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    pde_poisson_2d_dirichlet_matrix,
    poisson_2d_residual_norm,
    solve_laplace_2d_sor,
    solve_poisson_2d_sor,
)

## 五点差分矩阵

对内部网格点，$-\Delta u=f$ 的五点格式形成块三对角矩阵。每个主块来自同一条网格线，块之间由相邻网格线耦合。

In [ ]:
matrix = pde_poisson_2d_dirichlet_matrix(nx=3, ny=3, hx=0.25, hy=0.25)
matrix.shape, matrix[:3, :6]

## Laplace 方程

如果边界全为常数，Laplace 方程的内部解也应保持同一常数。

In [ ]:
laplace_boundary = np.ones((8, 8))
laplace = solve_laplace_2d_sor(laplace_boundary, hx=1.0 / 7.0, hy=1.0 / 7.0, omega=1.4, tolerance=1e-10)
laplace.iterations, laplace.residual_norm, laplace.solution[4, 4]

## Poisson 方程制造解

选择 $u=\sin(\pi x)\sin(\pi y)$，则 $-\Delta u=2\pi^2u$。这给出了检查边界、右端和残差的完整制造解。

In [ ]:
nx = ny = 18
x = np.linspace(0.0, 1.0, nx + 2)
y = np.linspace(0.0, 1.0, ny + 2)
xx, yy = np.meshgrid(x, y, indexing="ij")
exact = np.sin(math.pi * xx) * np.sin(math.pi * yy)
source = 2.0 * math.pi * math.pi * exact
boundary = np.zeros_like(exact)

poisson = solve_poisson_2d_sor(source, boundary, hx=x[1] - x[0], hy=y[1] - y[0], omega=1.6, tolerance=1e-9)
max_error = np.max(np.abs(poisson.solution - exact))
poisson.iterations, poisson.residual_norm, max_error

In [ ]:
poisson_2d_residual_norm(poisson.solution, source, x[1] - x[0], y[1] - y[0])

## 小结

* 椭圆型问题的边界条件直接决定内部解。
* 五点差分产生稀疏块三对角矩阵。
* SOR 通过松弛参数改善 Gauss-Seidel 迭代，残差是主要停止准则。
* 制造解能同时验证离散方程、边界处理和收敛结果。